In [ ]:
import os
kaggle = False
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'kaggle' in dirname:
            kaggle = True 
print(kaggle)

In [ ]:
import array
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import json
import bz2
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BertConfig, BertForPreTraining, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, IterableDataset
from accelerate import Accelerator
import time
import re
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import sys
from tqdm.auto import tqdm
from torchsummary import summary
import math
import datetime
from pathlib import Path

#print(keras.__version__)
# print(tf.__version__)

# if kaggle:
#     try:
#         tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
#         if tpu:
#             tf.config.experimental_connect_to_cluster(tpu)
#             tf.tpu.experimental.initialize_tpu_system(tpu)
#             strategy = tf.distribute.TPUStrategy(tpu)
#     except ValueError:
#         tpu = False
# print(tpu)

In [4]:
if kaggle:
    file_path = '/kaggle/input/pt-dataset/pt_dataset.json'
else:
    file_path = 'datasets/pt_dataset.json'
# file_path = 'combined.json'

def load_pretrain_data_json(file_path):
    urls = []
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            if len(data["url"]) < 200:
                urls.append(data["url"])
    return urls

def load_pretrain_data_bz2(file_path):
    urls = []
    with bz2.open(file_path, 'rt') as f:
        for line in f:
            data = json.loads(line)
            if len(data["url"]) < 200:
                urls.append(data["url"])
    return urls

In [5]:
MAX_LENGTH = 202
SEED = 42
UNIQUE_CHAR = ['!',	'#', '$', '&', "'", '(', ')', '*', '+', ',', '/', ':', ';', '=', '?', '@', '[', ']', '%',
               '-', '_', '~', '.',
               'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 
               'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 
               '0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
SPECIAL_TOKENS = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]', '[SEC]', '[ISEC]']

UNIQUE_CHAR_LEN = len(UNIQUE_CHAR)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

#print(UNIQUE_CHAR_LEN)

vocab =  UNIQUE_CHAR + SPECIAL_TOKENS
token_to_id  = {char: idx for idx, char in enumerate(vocab)}
id_to_token = {idx: char for char, idx in token_to_id.items()}
vocab_size = len(vocab)

In [ ]:
if '.bz2' in file_path:
    pretrain_urls1 = load_pretrain_data_bz2(file_path)
else:
    pretrain_urls1 = load_pretrain_data_json(file_path)
    
# pretrain_urls2 = [item.lower() for item in pretrain_urls1]

pretrain_urls, val_urls = train_test_split(
                                        # pretrain_urls2,
                                        pretrain_urls1, 
                                        test_size=0.05,
                                        random_state=42
)

del pretrain_urls1
# del pretrain_urls2

In [ ]:
print(len(pretrain_urls))
print(torch.cuda.is_available())

In [ ]:
# def sizeof_fmt(num, suffix='B'):
#     for unit in ['','Ki','Mi','Gi','Ti','Pi','Ei','Zi']:
#         if abs(num) < 1024.0:
#             return "%3.1f %s%s" % (num, unit, suffix)
#         num /= 1024.0
#     return "%.1f %s%s" % (num, 'Yi', suffix)
# for name, size in sorted(((name, sys.getsizeof(value)) for name, value in list(
#                         locals().items())), key= lambda x: -x[1])[:10]:
#     print("{:>30}: {:>8}".format(name, sizeof_fmt(size)))
    

# def get_real_size(obj):
#     """Recursively calculate the real memory size of an object."""
#     size = sys.getsizeof(obj)
#     if isinstance(obj, (list, tuple, set, frozenset)):
#         size += sum(get_real_size(item) for item in obj)
#     elif isinstance(obj, dict):
#         size += sum(get_real_size(key) + get_real_size(value) for key, value in obj.items())
#     return size

# def get_memory_usage():
#     # Separate user-defined and system variables
#     user_vars = {k: v for k, v in globals().items() if not k.startswith('_') and not callable(v)}
#     system_vars = {k: v for k, v in globals().items() if k.startswith('_') and not callable(v)}

#     # Calculate memory usage using custom get_real_size function
#     memory_usage = {k: get_real_size(v) for k, v in user_vars.items()}
#     system_memory = sum(get_real_size(v) for v in system_vars.values())

#     # Convert bytes to human-readable format
#     def sizeof_fmt(num, suffix='B'):
#         for unit in ['', 'K', 'M', 'G', 'T', 'P', 'E', 'Z']:
#             if abs(num) < 1024.0:
#                 return f"{num:3.1f}{unit}{suffix}"
#             num /= 1024.0
#         return f"{num:.1f}Yi{suffix}"

#     # Format memory usage
#     formatted_usage = {k: sizeof_fmt(v) for k, v in memory_usage.items()}
#     formatted_usage['_system_vars'] = sizeof_fmt(system_memory)

#     # Sort by size in descending order and extract values
#     sorted_sizes = sorted(formatted_usage.items(), key=lambda x: get_real_size(globals().get(x[0], 0)), reverse=True)

#     # Print the sorted list
#     for var, size in sorted_sizes:
#         print("{:>30}: {:>8}".format(var, size))

#     # Print totals
#     print("\nTotal Memory Usage:")
#     print(f"User Variables: {sizeof_fmt(sum(memory_usage.values()))}")
#     print(f"System Variables: {sizeof_fmt(system_memory)}")
#     print(f"Combined Total:     {sizeof_fmt(system_memory+sum(memory_usage.values()))}")

# # get_memory_usage()

# from collections import Counter
# import linecache
# import os
# import tracemalloc
# import resource

# def using(point=""):
#     usage=resource.getrusage(resource.RUSAGE_SELF)
#     return '''%s: usertime=%s systime=%s mem=%s mb
#            '''%(point,usage[0],usage[1],
#                 usage[2]/1024.0 )

# print(using("test"))

# def display_top(snapshot, key_type='lineno', limit=5):
#     snapshot = snapshot.filter_traces((
#         tracemalloc.Filter(False, "<frozen importlib._bootstrap>"),
#         tracemalloc.Filter(False, "<unknown>"),
#     ))
#     top_stats = snapshot.statistics(key_type)

#     print("Top %s lines" % limit)
#     for index, stat in enumerate(top_stats[:limit], 1):
#         frame = stat.traceback[0]
#         # replace "/path/to/module/file.py" with "module/file.py"
#         filename = os.sep.join(frame.filename.split(os.sep)[-2:])
#         print("#%s: %s:%s: %.1f KiB"
#               % (index, filename, frame.lineno, stat.size / 1024))
#         line = linecache.getline(frame.filename, frame.lineno).strip()
#         if line:
#             print('    %s' % line)

#     other = top_stats[limit:]
#     if other:
#         size = sum(stat.size for stat in other)
#         print("%s other: %.1f KiB" % (len(other), size / 1024))
#     total = sum(stat.size for stat in top_stats)
#     print("Total allocated size: %.1f KiB" % (total / 1024))


In [ ]:
# tracemalloc.start()
# def process_url(url):
#     parts = url.partition('//')[2].split('/')
#     return [part for part in parts if part][1:]

# def create_bag_parallel(urls):
#     print('Creating bag...')    
#     with ThreadPoolExecutor(max_workers=2) as executor:
#         bag = [token for tokens in executor.map(process_url, urls) for token in tokens]
#     print('Bag created')
#     return bag[:2300000]

def process_url(url):
    parts = url.partition('//')[2].partition('/')[2]
    if parts == '':
        return ''
    parts = parts.split('/')
    if parts[-1] == '':
        parts = parts[-2] + '/'
    else:
        parts = parts[-1]
    return parts


def create_bag_parallel(urls):
    print('Creating bag...')    
    with ThreadPoolExecutor(max_workers=2) as executor:
        bag = [result for result in executor.map(process_url, urls) if result]
    print('Bag created')
    return bag

# test = create_bag(pretrain_urls)
# print(test[:50])
# test2 = pd.DataFrame(test, columns=['sent'])
# test2 = test2.drop_duplicates()
# print(test2.count())
# print(test2.nunique())

# NSP Data Generator
def nsp_mlm_data_generator(urls, bag=None, mask_prob=0.15):
    # Create index pairs for NSP
    ## pairs = []
    is_next1 = 0
    not_next = 0
    
    if bag is None:    
        bag = create_bag_parallel(urls)

    bag_len = len(bag)
    # for i in range(len(urls)):
    #    counter3 = counter3 + 1
    #     if i % 10000 ==0:
    #         print(i/10000)
    for url in urls:
        x = url.partition('//')[2]
        is_secure = url.partition('//')[0]
        # is_secure = is_secure.partition('p')[2]        
        x = x.split('/')
        # test = x.rpartition('/')[0]
        yy = [c for c in x if c]
        del x
        
        # total = yy[0]
        # count = 1
        # while count < len(yy)-1:
        #     total = total + '/' + yy[count]
        #     count = count + 1
        
        if len(yy) == 1:
            total = yy[0]
        else:
            total = '/'.join(yy[:-1])
                    
        if len(yy) > 1 and is_next1 <= not_next:               
            ## pairs.append((total, yy[1], 1, is_secure)) 
            url2 = yy[-1]
            is_next = 1            
            is_next1 = is_next1 + 1
        
        elif len(yy) == 1:                  
            url2 = bag[np.random.randint(0, bag_len-1)]
            while (len(url2) + len(total)) > 199:
                url2 = bag[np.random.randint(0, bag_len-1)]            
            # pairs.append((total, j, 0, is_secure))     
            is_next = 0     
            not_next = not_next + 1
        else:
            url2 = bag[np.random.randint(0, bag_len-1)]
            while url2 == yy[-1] or (len(url2) + len(total)) > 199: 
                url2 = bag[np.random.randint(0, bag_len-1)]
            # pairs.append((total, j, 0, is_secure))  
            is_next = 0             
            not_next = not_next + 1

    # for url1, url2, is_next, is_secure in pairs:
        # Combine URLs with [SEP]
        if is_secure == 'https:':
            is_secure = '[SEC]'
        else:
            is_secure = '[ISEC]'
        tokens = ['[CLS]'] + [is_secure] + list(total) + ['[SEP]'] + list(url2) + ['[SEP]']

        if len(tokens) > MAX_LENGTH:
            tokens = tokens[:MAX_LENGTH-1] + [tokens[-1]]
            
        # Convert to token IDs
        input_ids = [token_to_id.get(token, token_to_id['[UNK]']) for token in tokens]
        
        # if token_to_id['[UNK]'] in input_ids:
        #     print('Unknown token: ', input_ids, tokens)
            
        # Create segment IDs (0 for first URL, 1 for second URL)
        token_type_ids = []
        found_sep = False
        for token in tokens:
            token_type_ids.append(1 if found_sep else 0)
            if token == "[SEP]":
                found_sep = True
        
        # Pad to MAX_LENGTH
        if len(input_ids) < MAX_LENGTH:
            padding = [token_to_id['[PAD]']] * (MAX_LENGTH - len(input_ids))
            input_ids += padding
            token_type_ids += [0] * (MAX_LENGTH - len(token_type_ids))
        
        # Create attention mask
        attention_mask = [1] * len(tokens) + [0] * (MAX_LENGTH - len(tokens))
        
        # MLM labels
        labels = input_ids.copy()
        masked_input = input_ids.copy()
        
        # Create masked version
        for i in range(len(input_ids)):
            token = id_to_token.get(input_ids[i], '[UNK]')
            
            # if '[UNK]' in token:
            #     print('Unknown token: ', token)
            
            if token in SPECIAL_TOKENS:
                labels[i] = -100  
                continue
            if attention_mask[i] and np.random.random() < mask_prob:
                labels[i] = input_ids[i]  # Store original ID
                # Apply masking strategy
                rand = np.random.random()
                if rand < 0.8:
                    masked_input[i] = token_to_id['[MASK]']
                elif rand < 0.9:
                    masked_input[i] = np.random.randint(len(UNIQUE_CHAR))
                else:
                    masked_input[i] = input_ids[i]  
            else:
                labels[i] = -100  # Ignore in loss

        yield {
            "input_ids": torch.tensor(masked_input),
            "token_type_ids": torch.tensor(token_type_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels),
            "next_sentence_label": torch.tensor(is_next)            
        }
    # get_memory_usage()

class NSPMLMDataset(IterableDataset):
    def __init__(self, urls, bag=None ): 
        super().__init__()
        self.urls = urls
        self.bag = bag        

    def __iter__(self):
        return nsp_mlm_data_generator(self.urls, bag=self.bag) 

full_bag = create_bag_parallel(pretrain_urls)

dataset = NSPMLMDataset(pretrain_urls, bag=full_bag)
val_dataset = NSPMLMDataset(val_urls, bag=full_bag)

batch_size = 128

pretrain_loader = DataLoader(
    # list(nsp_mlm_data_generator(pretrain_urls[:1000000])),
    dataset,
    batch_size=batch_size,
    pin_memory=True, 
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    pin_memory=True,
)

num_epochs = 10

# snapshot = tracemalloc.take_snapshot()
# tracemalloc.stop()
# display_top(snapshot)

# print('Not nsp', not_next)
# print('Is nsp', is_next1)

# check correct input shape
# print('prebatch')
# zxc = 0 
# for batch in pretrain_loader:
#     print(batch["input_ids"].shape) 
#     if zxc > 50:
#         break

# testasdf = create_bag_parallel(pretrain_urls)
# print(testasdf[1])
# print(len(testasdf))

In [ ]:
# Initialize accelerator
accelerator = Accelerator(mixed_precision='fp16')
device = accelerator.device

# Model configuration
config = BertConfig(
    vocab_size=vocab_size,
    hidden_size=512,                   #256
    num_hidden_layers=12,               #6
    num_attention_heads=8,             #8
    max_position_embeddings=MAX_LENGTH,
    type_vocab_size=2,  # For NSP segment IDs
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    intermediate_size=2048              
)

model = BertForPreTraining(config)

model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(), 
    # lr=1e-4,
    lr=5e-5,
    # betas=(0.9, 0.98),
    betas=(0.9, 0.999),
    weight_decay=0.01,
    )

total_steps = math.ceil(len(pretrain_urls) / batch_size) * num_epochs
warmup_steps = int(0.1 * total_steps)  
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
            
model, optimizer = accelerator.prepare(model, optimizer)
early_stopping = EarlyStopping(patience=2, min_delta=0.001)

print(device)

In [ ]:
# summary(model)

In [ ]:
# model, optimizer, pretrain_loader = accelerator.prepare(
#     model, optimizer, pretrain_loader
# )

# Training metrics storage
epoch_losses = []
epoch_accuracies = []
epoch_nsp_accuracies = []

epoch_val_losses = []
epoch_val_accuracies = []
epoch_val_nsp_accuracies = []

best_val_loss = float('inf')
start = time.time()

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total_masked = 0
    total_nsp_correct = 0
    num_batches = 0
    num_examples = 0    

    pbar = tqdm(pretrain_loader, total=math.ceil(len(pretrain_urls) / batch_size), desc=f"Train Epoch {epoch+1}", unit="batch")
    
    for batch in pbar:
        
        batch = {
            k: v.to(accelerator.device, non_blocking=True) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }
                
        # Move batch to GPU
        inputs = {
            "input_ids": batch["input_ids"],
            "token_type_ids": batch["token_type_ids"],
            "attention_mask": batch["attention_mask"],
            "labels": batch["labels"],
            "next_sentence_label": batch["next_sentence_label"]
        }
        
        # Forward pass
        outputs = model(**inputs)
        loss = outputs.loss

        # Backpropagation
        accelerator.backward(loss)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        # Calculate MLM accuracy
        with torch.no_grad():
            mlm_logits = outputs.prediction_logits
            mlm_preds = torch.argmax(mlm_logits, dim=-1)
            mask = inputs["labels"] != -100
            
            if mask.any():
                correct = mlm_preds[mask] == inputs["labels"][mask]
                total_correct += correct.sum().item()
                total_masked += mask.sum().item()
            
            # Calculate NSP accuracy
            nsp_logits = outputs.seq_relationship_logits
            nsp_preds = torch.argmax(nsp_logits, dim=-1)
            nsp_correct = nsp_preds == inputs["next_sentence_label"]
            total_nsp_correct += nsp_correct.sum().item()
        
        total_loss += loss.item()
        num_batches += 1
        num_examples += batch["input_ids"].size(0)
        
        pbar.set_postfix({
            "avg_loss": f"{(total_loss/num_batches):.4f}",
            "mlm_acc": f"{(total_correct/total_masked):.4f}",
            "nsp_acc": f"{(total_nsp_correct/num_examples):.4f}",
            "batches": f"{num_batches}"
        })        
        
    pbar.close()    
    
    avg_loss = total_loss / num_batches if num_batches else 0.0    
    mlm_accuracy = total_correct / total_masked if total_masked > 0 else 0
    nsp_accuracy = total_nsp_correct / num_examples if num_examples > 0 else 0.0
    
    print(f"Epoch {epoch+1} | Total Loss: {avg_loss:.4f} | MLM Acc: {mlm_accuracy:.4f} | NSP Acc: {nsp_accuracy:.4f}")

    epoch_losses.append(avg_loss)
    epoch_accuracies.append(mlm_accuracy)
    epoch_nsp_accuracies.append(nsp_accuracy)        
        
    model.eval()
    total_val_loss = 0
    total_val_mlm_correct = 0
    total_val_masked = 0
    total_val_nsp_correct = 0
    num_val_batches = 0
    num_val_examples = 0
    
    with torch.no_grad():
        pbar2 = tqdm(val_loader, total=math.ceil(len(val_urls) / batch_size), desc=f" Validation Epoch {epoch+1}", unit="batch")
        for batch in pbar2:
            batch = {
                k: v.to(accelerator.device, non_blocking=True) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }
            
            # Prepare inputs
            inputs = {
                "input_ids": batch["input_ids"],
                "token_type_ids": batch["token_type_ids"],
                "attention_mask": batch["attention_mask"],
                "labels": batch["labels"],
                "next_sentence_label": batch["next_sentence_label"]
            }
            
            # Forward pass
            outputs = model(**inputs)
            loss = outputs.loss
            
            # Calculate MLM accuracy
            mlm_logits = outputs.prediction_logits
            mlm_preds = torch.argmax(mlm_logits, dim=-1)
            mask = inputs["labels"] != -100
            
            if mask.any():
                correct = mlm_preds[mask] == inputs["labels"][mask]
                total_val_mlm_correct += correct.sum().item()
                total_val_masked += mask.sum().item()
            
            # Calculate NSP accuracy
            nsp_logits = outputs.seq_relationship_logits
            nsp_preds = torch.argmax(nsp_logits, dim=-1)
            nsp_correct = nsp_preds == inputs["next_sentence_label"]
            total_val_nsp_correct += nsp_correct.sum().item()
            
            # Update statistics
            total_val_loss += loss.item()
            num_val_batches += 1
            num_val_examples += batch["input_ids"].size(0)

            pbar2.set_postfix({
                "avg_val_loss": f"{(total_val_loss/num_val_batches):.4f}",
                "mlm_val_acc": f"{(total_val_mlm_correct/total_val_masked):.4f}",
                "nsp_val_acc": f"{(total_val_nsp_correct/num_val_examples):.4f}",                
                "batches": f"{num_val_batches}"
            })        
        
    pbar2.close()              
    
    avg_val_loss = total_val_loss / num_val_batches if num_val_batches else 0.0
    val_mlm_accuracy = total_val_mlm_correct / total_val_masked if total_val_masked > 0 else 0
    val_nsp_accuracy = total_val_nsp_correct / num_val_examples if num_val_examples > 0 else 0.0
    
    print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | Val MLM Acc: {val_mlm_accuracy:.4f} | Val NSP Acc: {val_nsp_accuracy:.4f}")

    epoch_val_losses.append(avg_val_loss)
    epoch_val_accuracies.append(val_mlm_accuracy)
    epoch_val_nsp_accuracies.append(val_nsp_accuracy)        
    
    # if avg_val_loss < best_val_loss:
    #     best_val_loss = avg_val_loss
    #     accelerator.save_state('best_model.pt')
        
    early_stopping(avg_val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break

# Save final model
# if kaggle:
#     accelerator.save_state('/kaggle/working/pretrain.pt')
#     Path("/kaggle/working/best_model.pt").rename("/kaggle/working/pretrain.pt/best_model.pt")
# else:    
#     accelerator.save_state('pretrain.pt')
#     try:
#         Path("best_model.pt").rename("pretrain.pt/best_model.pt")
#     except:
#         print("Could not move best_model.pt")


end = time.time()
print('Time taken: ', (end - start)/60, ' minutes')

In [13]:
i = 0
if kaggle:
    try:
        with open("/kaggle/working/pretrain.pt/stats.txt", "x") as f:
            while i < len(epoch_losses):
                f.write(f"Epoch {i+1} | Train Loss: {epoch_losses[i]:.4f} | Train MLM Acc: {epoch_accuracies[i]:.4f} | Train NSP Acc: {epoch_nsp_accuracies[i]:.4f}\n")
                f.write(f"Epoch {i+1} | Val Loss: {epoch_val_losses[i]:.4f} | Val MLM Acc: {epoch_val_accuracies[i]:.4f} | Val NSP Acc: {epoch_val_nsp_accuracies[i]:.4f}\n")
                i = i + 1
    except FileExistsError:
        with open("/kaggle/working/pretrain.pt/stats.txt", "w") as f:
            while i < len(epoch_losses):
                f.write(f"Epoch {i+1} | Train Loss: {epoch_losses[i]:.4f} | Train MLM Acc: {epoch_accuracies[i]:.4f} | Train NSP Acc: {epoch_nsp_accuracies[i]:.4f}\n")
                f.write(f"Epoch {i+1} | Val Loss: {epoch_val_losses[i]:.4f} | Val MLM Acc: {epoch_val_accuracies[i]:.4f} | Val NSP Acc: {epoch_val_nsp_accuracies[i]:.4f}\n")
                i = i + 1                
else:    
    try:
        with open("pretrain.pt/stats.txt", "x") as f:
            while i < len(epoch_losses):
                f.write(f"Epoch {i+1} | Train Loss: {epoch_losses[i]:.4f} | Train MLM Acc: {epoch_accuracies[i]:.4f} | Train NSP Acc: {epoch_nsp_accuracies[i]:.4f}\n")
                f.write(f"Epoch {i+1} | Val Loss: {epoch_val_losses[i]:.4f} | Val MLM Acc: {epoch_val_accuracies[i]:.4f} | Val NSP Acc: {epoch_val_nsp_accuracies[i]:.4f}\n")
                i = i + 1          
    except FileExistsError:
        with open("pretrain.pt/stats.txt", "w") as f:
            while i < len(epoch_losses):
                f.write(f"Epoch {i+1} | Train Loss: {epoch_losses[i]:.4f} | Train MLM Acc: {epoch_accuracies[i]:.4f} | Train NSP Acc: {epoch_nsp_accuracies[i]:.4f}\n")
                f.write(f"Epoch {i+1} | Val Loss: {epoch_val_losses[i]:.4f} | Val MLM Acc: {epoch_val_accuracies[i]:.4f} | Val NSP Acc: {epoch_val_nsp_accuracies[i]:.4f}\n")
                i = i + 1               


In [ ]:
# Plot training metrics
plt.figure(figsize=(15, 10))

# Loss plot
plt.subplot(2, 2, 1)
plt.plot(range(1, len(epoch_losses)+1), epoch_losses, 'bo-', label='Train Loss')
plt.plot(range(1, len(epoch_val_losses)+1), epoch_val_losses, 'ro-', label='Validation Loss')
plt.title('Training Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xlim(0, len(epoch_losses)+1)
plt.ylim(0, max(max(epoch_losses), max(epoch_val_losses)) * 1.1)
plt.legend()
plt.grid(True)

# MLM Accuracy plot
plt.subplot(2, 2, 3)
plt.plot(range(1, len(epoch_accuracies)+1), epoch_accuracies, 'bo-', label='MLM Train Accuracy')
plt.plot(range(1, len(epoch_val_accuracies)+1), epoch_val_accuracies, 'ro-', label='MLM Validation Accuracy')
plt.title('MLM Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xlim(0, len(epoch_val_accuracies)+1)
plt.legend()
plt.grid(True)

# NSP Accuracy plot
plt.subplot(2, 2, 2)
plt.plot(range(1, len(epoch_nsp_accuracies)+1), epoch_nsp_accuracies, 'bo-', label='NSP Train Accuracy')
plt.plot(range(1, len(epoch_val_nsp_accuracies)+1), epoch_val_nsp_accuracies, 'ro-', label='NSP Validation Accuracy')
plt.title('NSP Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xlim(0, len(epoch_nsp_accuracies)+1)
plt.legend()
plt.grid(True)

plt.tight_layout()
if kaggle:
    plt.savefig('/kaggle/working/pretrain.pt/training_metrics.png')
else: 
    plt.savefig('run4_pretrain.pt/training_metrics_fix.png')
plt.show()

print("Train MLM accuracy: ", epoch_accuracies)
print("Validation MLM accuracy: ", epoch_val_accuracies)
print("Train NSP accuracy: ", epoch_nsp_accuracies)
print("Validation NSP accuracy: ", epoch_val_nsp_accuracies)
print("Train loss accuracy: ", epoch_losses)
print("Validation loss accuracy: ", epoch_val_losses)

In [ ]:
# for epoch in range(1):
#     model.train()
#     total_loss = 0
#     total_correct = 0
#     total_masked = 0
#     total_nsp_correct = 0
#     num_batches = 0
#     num_examples = 0    

#     pbar = tqdm(pretrain_loader, total=math.ceil(len(pretrain_urls) / batch_size), desc=f"Train Epoch {epoch+1}", unit="batch")
    
#     for batch in pbar:
        
#         batch = {
#             k: v.to(accelerator.device, non_blocking=True) if torch.is_tensor(v) else v
#             for k, v in batch.items()
#         }
                
#         # Move batch to GPU
#         inputs = {
#             "input_ids": batch["input_ids"],
#             "token_type_ids": batch["token_type_ids"],
#             "attention_mask": batch["attention_mask"],
#             "labels": batch["labels"],
#             "next_sentence_label": batch["next_sentence_label"]
#         }
        
#         # Forward pass
#         outputs = model(**inputs)
#         loss = outputs.loss

#         # Backpropagation
#         accelerator.backward(loss)
#         optimizer.step()
#         scheduler.step()
#         optimizer.zero_grad()
        
#         # Calculate MLM accuracy
#         with torch.no_grad():
#             mlm_logits = outputs.prediction_logits
#             mlm_preds = torch.argmax(mlm_logits, dim=-1)
#             mask = inputs["labels"] != -100
            
#             if mask.any():
#                 correct = mlm_preds[mask] == inputs["labels"][mask]
#                 total_correct += correct.sum().item()
#                 total_masked += mask.sum().item()
            
#             # Calculate NSP accuracy
#             nsp_logits = outputs.seq_relationship_logits
#             nsp_preds = torch.argmax(nsp_logits, dim=-1)
#             nsp_correct = nsp_preds == inputs["next_sentence_label"]
#             total_nsp_correct += nsp_correct.sum().item()
        
#         total_loss += loss.item()
#         num_batches += 1
#         num_examples += batch["input_ids"].size(0)
        
#         pbar.set_postfix({
#             "avg_loss": f"{(total_loss/num_batches):.4f}",
#             "mlm_acc": f"{(total_correct/total_masked):.4f}",
#             "nsp_acc": f"{(total_nsp_correct/num_examples):.4f}",
#             "batches": f"{num_batches}"
#         })        
        
#     pbar.close()    
    
#     avg_loss = total_loss / num_batches if num_batches else 0.0    
#     mlm_accuracy = total_correct / total_masked if total_masked > 0 else 0
#     nsp_accuracy = total_nsp_correct / num_examples if num_examples > 0 else 0.0
    
#     print(f"Epoch {epoch+1} | Total Loss: {avg_loss:.4f} | MLM Acc: {mlm_accuracy:.4f} | NSP Acc: {nsp_accuracy:.4f}")

#     epoch_losses.append(avg_loss)
#     epoch_accuracies.append(mlm_accuracy)
#     epoch_nsp_accuracies.append(nsp_accuracy)        
        
#     model.eval()
#     total_val_loss = 0
#     total_val_mlm_correct = 0
#     total_val_masked = 0
#     total_val_nsp_correct = 0
#     num_val_batches = 0
#     num_val_examples = 0
    
#     with torch.no_grad():
#         pbar2 = tqdm(val_loader, total=math.ceil(len(val_urls) / batch_size), desc=f" Validation Epoch {epoch+1}", unit="batch")
#         for batch in pbar2:
#             batch = {
#                 k: v.to(accelerator.device, non_blocking=True) if torch.is_tensor(v) else v
#                 for k, v in batch.items()
#             }
            
#             # Prepare inputs
#             inputs = {
#                 "input_ids": batch["input_ids"],
#                 "token_type_ids": batch["token_type_ids"],
#                 "attention_mask": batch["attention_mask"],
#                 "labels": batch["labels"],
#                 "next_sentence_label": batch["next_sentence_label"]
#             }
            
#             # Forward pass
#             outputs = model(**inputs)
#             loss = outputs.loss
            
#             # Calculate MLM accuracy
#             mlm_logits = outputs.prediction_logits
#             mlm_preds = torch.argmax(mlm_logits, dim=-1)
#             mask = inputs["labels"] != -100
            
#             if mask.any():
#                 correct = mlm_preds[mask] == inputs["labels"][mask]
#                 total_val_mlm_correct += correct.sum().item()
#                 total_val_masked += mask.sum().item()
            
#             # Calculate NSP accuracy
#             nsp_logits = outputs.seq_relationship_logits
#             nsp_preds = torch.argmax(nsp_logits, dim=-1)
#             nsp_correct = nsp_preds == inputs["next_sentence_label"]
#             total_val_nsp_correct += nsp_correct.sum().item()
            
#             # Update statistics
#             total_val_loss += loss.item()
#             num_val_batches += 1
#             num_val_examples += batch["input_ids"].size(0)

#             pbar2.set_postfix({
#                 "avg_val_loss": f"{(total_val_loss/num_val_batches):.4f}",
#                 "mlm_val_acc": f"{(total_val_mlm_correct/total_val_masked):.4f}",
#                 "nsp_val_acc": f"{(total_val_nsp_correct/num_val_examples):.4f}",                
#                 "batches": f"{num_val_batches}"
#             })        
        
#     pbar2.close()              
    
#     avg_val_loss = total_val_loss / num_val_batches if num_val_batches else 0.0
#     val_mlm_accuracy = total_val_mlm_correct / total_val_masked if total_val_masked > 0 else 0
#     val_nsp_accuracy = total_val_nsp_correct / num_val_examples if num_val_examples > 0 else 0.0
    
#     print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | Val MLM Acc: {val_mlm_accuracy:.4f} | Val NSP Acc: {val_nsp_accuracy:.4f}")

#     epoch_val_losses.append(avg_val_loss)
#     epoch_val_accuracies.append(val_mlm_accuracy)
#     epoch_val_nsp_accuracies.append(val_nsp_accuracy)        
    
#     if avg_val_loss < best_val_loss:
#         best_val_loss = avg_val_loss
#         accelerator.save_state('best_model.pt')
        
#     early_stopping(avg_val_loss)
#     if early_stopping.early_stop:
#         print("Early stopping triggered")
#         break

# # Save final model
# # if kaggle:
# #     accelerator.save_state('/kaggle/working/pretrain.pt')
# #     Path("/kaggle/working/best_model.pt").rename("/kaggle/working/pretrain.pt/best_model.pt")
# # else:    
# #     accelerator.save_state('pretrain.pt')
# #     Path("best_model.pt").rename("pretrain.pt/best_model.pt")